In [ ]:
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI
from IPython.display import Markdown, display
from LoadProperties import LoadProperties

# Tools
calculator = lambda x: str(eval(x)) if x.strip() else "Invalid"
get_founder = lambda x: {"Apple": "Steve Jobs", "Microsoft": "Bill Gates", "Amazon": "Jeff Bezos", "Google": "Larry Page and Sergey Brin"}.get(x.strip(), "Founder not found.")
get_city_info = lambda x: {"Paris": "Paris is the capital of France.", "New York": "New York is known as the Big Apple.", "Tokyo": "Tokyo is the capital of Japan."}.get(x.strip(), "City info not found.")
tools = [
    Tool("Calculator", calculator, "Solve math like '45 * 3'."),
    Tool("CompanyFounderLookup", get_founder, "Find the founder of a company."),
    Tool("CityInformation", get_city_info, "Get facts about cities.")
]

# Output schema
parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="answer", description="Final answer."),
    ResponseSchema(name="source", description="Tool used.")
])
fmt = parser.get_format_instructions()

# LLM
props = LoadProperties()
llm = ChatOCIGenAI(
    model_id='meta.llama-3.3-70b-instruct',
    service_endpoint=props.getEndpoint(),
    compartment_id=props.getCompartment(),
    auth_type='INSTANCE_PRINCIPAL',
    model_kwargs={"max_tokens": 600}
)

# Agent
system_prompt =f"""
You are a helpful agent that uses tools to answer user questions. Choose the correct tool to respond, and keep your answers concise and accurate.

You have access to:
- Calculator: Solve basic math like '45 * 3'
- CompanyFounderLookup: Find the founder of companies like Apple, Microsoft
- CityInformation: Get facts about cities like Paris, Tokyo

Here are some examples of how to answer:

Q: What is 45 * 3?
A: 135 (Calculator)

Q: Who founded Amazon?
A: Jeff Bezos (CompanyFounderLookup)

Q: Tell me something about Tokyo.
A: Tokyo is the capital of Japan. (CityInformation)

Always pick the most suitable tool and return only the final answer with the tool name in parentheses.
"""
agent = initialize_agent(
    tools=tools, llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, handle_parsing_errors=True,
    agent_kwargs={"system_message": system_prompt},
    max_iterations=4, early_stopping_method="generate"
)

# Run queries
queries = ["What is 45 * 3?", "Who founded Google?", "Tell me something about Paris."]
# Replace the loop with this:
for q in queries:
    print(f"\n {q}")
    res = agent.invoke({"input": q})
    answer = res.get("output", " No output")
    display(Markdown(f"**Answer:** {answer}"))
